In [ ]:

# ============================================================
# SECTION 1: IMPORTS AND CONFIGURATION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from pathlib import Path
from PIL import Image

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# Paths
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "preprocessed"

MODEL_PATH = PROJECT_ROOT / "models" / "baseline_unet_best.pth"

RESULTS_DIR = PROJECT_ROOT / "results" / "visualizations"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
IMAGE_SIZE = 256
THRESHOLD = 0.5

print("Device:", DEVICE)
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Checkpoint:", MODEL_PATH)
print("Visualization results:", RESULTS_DIR)

assert DATA_DIR.exists()
assert MODEL_PATH.exists()

print("\nVisualization configuration loaded.")


In [ ]:

# ============================================================
# SECTION 2: BASELINE U-NET ARCHITECTURE
# ============================================================

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):

    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        self.bottleneck = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = DoubleConv(64, 32)

        self.output = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


# Initialize and load trained model
model = UNet().to(DEVICE)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())

print("Baseline U-Net loaded successfully.")
print("Parameters:", f"{total_params:,}")
print("Best epoch:", checkpoint["epoch"])
print("Validation Dice:", checkpoint["best_val_dice"])


In [ ]:

# ============================================================
# SECTION 3: LOAD TEST DATA AND METADATA
# ============================================================

class BUSIDataset(torch.utils.data.Dataset):

    def __init__(self, split_dir):
        self.image_dir = split_dir / "images"
        self.mask_dir = split_dir / "masks"

        self.image_paths = sorted(self.image_dir.glob("*.png"))

        assert len(self.image_paths) > 0

        for image_path in self.image_paths:
            assert (self.mask_dir / image_path.name).exists()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]
        mask_path = self.mask_dir / image_path.name

        image = Image.open(image_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.float32)

        mask = (mask > 0).astype(np.float32)

        image_tensor = torch.from_numpy(image).unsqueeze(0)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)

        return image_tensor, mask_tensor


# Load test dataset
test_dataset = BUSIDataset(DATA_DIR / "test")

# Load diagnostic categories
splits_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "dataset_splits.csv"
)

test_metadata = splits_df[
    splits_df["split"] == "test"
][["filename", "category"]].copy()

# Map metadata to dataset filenames
test_metadata["dataset_filename"] = (
    test_metadata["filename"]
    .str.replace(r"^(benign|malignant|normal)_", "", regex=True)
)

test_metadata = test_metadata.set_index("dataset_filename")

print("Test images:", len(test_dataset))
print("\nCategory distribution:")
print(test_metadata["category"].value_counts())

# Verify mapping
dataset_filenames = [
    path.name for path in test_dataset.image_paths
]

unmatched = set(dataset_filenames) - set(test_metadata.index)

print("\nUnmatched images:", len(unmatched))

assert len(unmatched) == 0, "Some test images lack category metadata."


In [ ]:

# ============================================================
# SECTION 3: LOAD TEST DATA AND METADATA
# ============================================================

class BUSIDataset(torch.utils.data.Dataset):

    def __init__(self, split_dir):
        self.image_dir = split_dir / "images"
        self.mask_dir = split_dir / "masks"

        self.image_paths = sorted(self.image_dir.glob("*.png"))

        assert len(self.image_paths) > 0

        for image_path in self.image_paths:
            assert (self.mask_dir / image_path.name).exists()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]
        mask_path = self.mask_dir / image_path.name

        image = Image.open(image_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.float32)

        mask = (mask > 0).astype(np.float32)

        image_tensor = torch.from_numpy(image).unsqueeze(0)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)

        return image_tensor, mask_tensor


# Load test dataset
test_dataset = BUSIDataset(DATA_DIR / "test")

# Load diagnostic categories
splits_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "dataset_splits.csv"
)

test_metadata = splits_df[
    splits_df["split"] == "test"
][["filename", "category"]].copy()

# Reconstruct processed dataset filenames
test_metadata["dataset_filename"] = (
    test_metadata["category"] + "_" + test_metadata["filename"]
)

test_metadata = test_metadata.set_index("dataset_filename")

# Verify mapping
dataset_filenames = [
    path.name for path in test_dataset.image_paths
]

unmatched = set(dataset_filenames) - set(test_metadata.index)

print("Test images:", len(test_dataset))

print("\nCategory distribution:")
print(test_metadata["category"].value_counts())

print("\nUnmatched images:", len(unmatched))

assert len(unmatched) == 0, "Some test images lack category metadata."

print("\nAll test images mapped successfully.")


In [ ]:

# ============================================================
# SECTION 4: GENERATE TEST SET PREDICTIONS
# ============================================================

prediction_records = []

model.eval()

with torch.no_grad():

    for idx in range(len(test_dataset)):

        image, mask = test_dataset[idx]

        input_tensor = image.unsqueeze(0).to(DEVICE)

        logits = model(input_tensor)
        probability = torch.sigmoid(logits)[0, 0].cpu().numpy()

        prediction = (probability >= THRESHOLD).astype(np.uint8)

        filename = test_dataset.image_paths[idx].name
        category = test_metadata.loc[filename, "category"]

        prediction_records.append({
            "image_name": filename,
            "category": category,
            "image": image[0].numpy(),
            "ground_truth": mask[0].numpy(),
            "prediction": prediction,
            "probability": probability
        })

print("Predictions generated:", len(prediction_records))

print("\nPredictions by category:")

prediction_categories = pd.Series([
    record["category"] for record in prediction_records
])

print(prediction_categories.value_counts())

# Verify output dimensions
sample = prediction_records[0]

print("\nSample verification:")
print("Image shape:", sample["image"].shape)
print("Ground truth shape:", sample["ground_truth"].shape)
print("Prediction shape:", sample["prediction"].shape)
print("Probability shape:", sample["probability"].shape)
print("Prediction values:", np.unique(sample["prediction"]))


In [ ]:
def visualize_prediction(record, save=True):

    image = record["image"]
    ground_truth = record["ground_truth"]
    prediction = record["prediction"]

    true_positive = (ground_truth == 1) & (prediction == 1)
    false_positive = (ground_truth == 0) & (prediction == 1)
    false_negative = (ground_truth == 1) & (prediction == 0)

    overlay = np.zeros(
        (*ground_truth.shape, 3),
        dtype=np.float32
    )

    overlay[true_positive] = [0, 1, 0]
    overlay[false_positive] = [1, 0, 0]
    overlay[false_negative] = [0, 0, 1]

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Original Image")

    axes[1].imshow(ground_truth, cmap="gray")
    axes[1].set_title("Ground Truth")

    axes[2].imshow(prediction, cmap="gray")
    axes[2].set_title("U-Net Prediction")

    axes[3].imshow(image, cmap="gray")
    axes[3].imshow(overlay, alpha=0.55)
    axes[3].set_title("Segmentation Overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"{record['category'].upper()} | {record['image_name']}"
    )

    plt.tight_layout()

    if save:
        output_path = (
            RESULTS_DIR /
            f"{Path(record['image_name']).stem}_comparison.png"
        )

        plt.savefig(
            output_path,
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()


benign_example = next(
    record for record in prediction_records
    if record["category"] == "benign"
)

visualize_prediction(benign_example)

In [ ]:
malignant_example = next(
    record for record in prediction_records
    if record["category"] == "malignant"
)

normal_example = next(
    record for record in prediction_records
    if record["category"] == "normal"
)

visualize_prediction(malignant_example)
visualize_prediction(normal_example)

In [ ]:
def dice_score(gt, pred, eps=1e-8):
    gt = gt.astype(bool)
    pred = pred.astype(bool)
    intersection = (gt & pred).sum()
    return (2 * intersection) / (gt.sum() + pred.sum() + eps)


rows = []
for record in prediction_records:
    gt = record["ground_truth"]
    pred = record["prediction"]

    rows.append({
        "image_name": record["image_name"],
        "category": record["category"],
        "dice": dice_score(gt, pred) if record["category"] != "normal" else np.nan,
        "gt_pixels": int(gt.sum()),
        "pred_pixels": int(pred.sum()),
    })

viz_metrics = pd.DataFrame(rows)

lesion_metrics = viz_metrics[viz_metrics["category"] != "normal"]

print(f"Lesion images: {len(lesion_metrics)}")
print(f"Mean lesion Dice: {lesion_metrics['dice'].mean():.4f}")
print()
print(lesion_metrics.groupby("category")["dice"].agg(["count", "mean"]).round(4))

In [ ]:
selected_cases = {}

for category in ["benign", "malignant"]:
    category_df = (
        lesion_metrics[lesion_metrics["category"] == category]
        .sort_values("dice")
        .reset_index(drop=True)
    )

    selected_cases[f"{category}_worst"] = category_df.iloc[0]
    selected_cases[f"{category}_median"] = category_df.iloc[len(category_df) // 2]
    selected_cases[f"{category}_best"] = category_df.iloc[-1]

selection_table = pd.DataFrame(selected_cases).T[
    ["image_name", "category", "dice", "gt_pixels", "pred_pixels"]
]
selection_table["dice"] = selection_table["dice"].astype(float).round(4)

print("Representative lesion cases:")
print(selection_table.to_string())

normal_df = viz_metrics[viz_metrics["category"] == "normal"]
normal_fp = normal_df[normal_df["pred_pixels"] > 0].sort_values(
    "pred_pixels", ascending=False
)

print()
print(f"Normal images with any predicted pixels: {len(normal_fp)} of {len(normal_df)}")
print(normal_fp[["image_name", "pred_pixels"]].to_string(index=False))

In [ ]:
record_lookup = {record["image_name"]: record for record in prediction_records}

for key, row in selection_table.iterrows():
    print(f"{key}: Dice = {row['dice']:.4f}")
    visualize_prediction(record_lookup[row["image_name"]])

In [ ]:
from matplotlib.patches import Patch


def make_rgba_overlay(record, alpha=0.55):
    gt = record["ground_truth"]
    pred = record["prediction"]

    rgba = np.zeros((*gt.shape, 4), dtype=np.float32)

    tp = (gt == 1) & (pred == 1)
    fp = (gt == 0) & (pred == 1)
    fn = (gt == 1) & (pred == 0)

    rgba[tp, :3] = [0, 1, 0]
    rgba[fp, :3] = [1, 0, 0]
    rgba[fn, :3] = [0, 0, 1]
    rgba[tp | fp | fn, 3] = alpha

    return rgba


grid_keys = [
    ["benign_worst", "benign_median", "benign_best"],
    ["malignant_worst", "malignant_median", "malignant_best"],
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10.5))

for i, row_keys in enumerate(grid_keys):
    for j, key in enumerate(row_keys):
        info = selection_table.loc[key]
        record = record_lookup[info["image_name"]]

        ax = axes[i, j]
        ax.imshow(record["image"], cmap="gray")
        ax.imshow(make_rgba_overlay(record))
        ax.set_title(
            f"{info['category'].capitalize()} - {key.split('_')[1]}\n"
            f"{info['image_name']} | Dice = {info['dice']:.3f}",
            fontsize=11
        )
        ax.axis("off")

legend_handles = [
    Patch(facecolor="green", label="True positive"),
    Patch(facecolor="red", label="False positive"),
    Patch(facecolor="blue", label="False negative"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, fontsize=12, frameon=False)

fig.suptitle("Baseline U-Net: representative test cases by Dice", fontsize=14)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])

montage_path = RESULTS_DIR / "baseline_representative_montage.png"
plt.savefig(montage_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {montage_path}")

In [ ]:
from matplotlib.patches import Patch


def make_rgba_overlay(record, alpha=0.55):
    gt = record["ground_truth"]
    pred = record["prediction"]

    rgba = np.zeros((*gt.shape, 4), dtype=np.float32)

    tp = (gt == 1) & (pred == 1)
    fp = (gt == 0) & (pred == 1)
    fn = (gt == 1) & (pred == 0)

    rgba[tp, :3] = [0, 1, 0]
    rgba[fp, :3] = [1, 0, 0]
    rgba[fn, :3] = [0, 0, 1]
    rgba[tp | fp | fn, 3] = alpha

    return rgba


grid_keys = [
    ["benign_worst", "benign_median", "benign_best"],
    ["malignant_worst", "malignant_median", "malignant_best"],
]

fig, axes = plt.subplots(2, 3, figsize=(15, 11.5))

for i, row_keys in enumerate(grid_keys):
    for j, key in enumerate(row_keys):
        info = selection_table.loc[key]
        record = record_lookup[info["image_name"]]

        ax = axes[i, j]
        ax.imshow(record["image"], cmap="gray")
        ax.imshow(make_rgba_overlay(record))
        ax.set_title(
            f"{info['category'].capitalize()} - {key.split('_')[1]}\n"
            f"{info['image_name']} | Dice = {info['dice']:.3f}",
            fontsize=11,
            pad=10
        )
        ax.axis("off")

# Explicit spacing instead of tight_layout so titles are never clipped
fig.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.90,
    bottom=0.08,
    hspace=0.22,
    wspace=0.05
)

fig.suptitle(
    "Baseline U-Net: representative test cases by Dice",
    fontsize=14,
    y=0.97
)

legend_handles = [
    Patch(facecolor="green", label="True positive"),
    Patch(facecolor="red", label="False positive"),
    Patch(facecolor="blue", label="False negative"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    fontsize=12,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01)
)

montage_path = RESULTS_DIR / "baseline_representative_montage.png"
fig.savefig(montage_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved: {montage_path}")

In [ ]:
top_fp = normal_fp.head(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))

for ax, (_, row) in zip(axes, top_fp.iterrows()):
    record = record_lookup[row["image_name"]]

    ax.imshow(record["image"], cmap="gray")
    ax.imshow(make_rgba_overlay(record))
    ax.set_title(
        f"{row['image_name']}\nFalse-positive pixels = {row['pred_pixels']}",
        fontsize=11
    )
    ax.axis("off")

fig.suptitle(
    "Baseline U-Net: largest false positives on normal test images (red)",
    fontsize=14
)
fig.tight_layout(rect=[0, 0, 1, 0.95])

fp_path = RESULTS_DIR / "baseline_normal_false_positives.png"
plt.savefig(fp_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {fp_path}")